# Le jeu de données IMDB

Dans ce notebook, vous allez utiliser le jeu de données imdb : c'est un ensemble de 50 000 critiques hautement polarisées de la base de données Internet Movie Database. Les critiques sont réparties en 25 000 critiques pour l'ensemble d'entraînement et 25 000 critiques pour l'ensemble de test. Chaque ensemble contient 50% de critiques négatives et 50% de critiques positives. 

Tout comme le jeu de données MNIST, le jeu de données IMDB est fourni avec la bibliothèque Keras. Il est déjà pré-traité : les critiques de films (des séquences de mots) ont été transformées en séquences d'entiers, où chaque entier représente un mot spécifique dans un dictionnaire.



# 1. Télécharger les données

Le code suivant permet de télécharger le jeu de données (environ 80Mo de données qui seront téléchargées sur votre machine). 

<blockquote>
A noter que l'argument `num_words` signifie que vous ne conservez que 10 000 mots les plus fréquentes dans l'ensemble d'entraînement. Les mots rares seront supprimés. Cela vous permet de travailler avec des données vectorielles de taille raisonnable.  

Les variables `x_train` et `x_test` contiennent des listes de critiques, chaque critique est une liste d'indices de mots. 

Les variables `y_train` et `y_test`contiennt des listes de 0 et 1, où 0 correspond à une critique négative et 1 à une critique positive.  
</blockquote>

In [ ]:
from keras.datasets import imdb
from keras import preprocessing

max_features = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# 2. Embedding de mots

Une des méthodes populaires et puissantes pour associer un vecteur à un mot est d'utiliser des vecteurs de mots denses, également appelés word embeddings. 
Il y'a deux méthodes pour obtenir des embeddings de mots : 

* Apprendre les embeddings de mots en même temps que la tâche principale à laquelle vous intéressez. 
* Utiliser un modèle pré-entraîné comme Word2Vec ou GloVe. 

Détaillons ces deux méthodes :

### 2.1 Apprendre des embeddings de mots avec la couche Embedding

La couche embedding est comme un dictionnaire qui associe, aux indices entiers, des vecteurs denses. Il s'agit en fait d'une recherche dans un dictionnaire. 

Indice du mot -> Couche Embedding -> Vecteur correspondant au mot

La couche Embedding prend en entrée un tenseur de 2D d'entiers de la forme (sample, sequence_length), où chaque entrée est une séquence d'entiers. Elle peut également traiter des séquences de longueurs variables : par exemple, vous pourriez alimenter la couche Embedding avec des lots d'exemples de la forme (32, 10) ou de la forme (64,15). Toutes les séquences d'un lot doivent cependant avoir la même longueur, par ce que vous devez les regrouper en un seul tenseur. 
Les séquences qui sont donc plus courtes que les autres doivent être remplies de zéros, et les séquences qui sont plus longues doivent être tronquées. 

Cette couche retourne un tenseur de 3D en virgule flottante de la forme (samples, sequence_length, enbedding_dimentionnality). Un tel tenseur 3D peut ensuite être traité par un RNN. 

#### 2.1.1 Tronquer les critiques en 20 mots

In [32]:
maxlen = 20
x_train = preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

#### 2.1.2 Construire le modèle RNN

Le modèle est constituer de 3 couches :

<ul>
    <li> <strong>Couche Emebbedding :</strong> le modèle apprendra des embeddings en 8 dimensions pour chacun des 10000 mots. Il transformera les séquences d'netiers (tenseur 2D) en séquences embedded (tenseur 3D) </li>
    <li> <strong>Couche RNN:</strong> Une couche de type RNN. TensorFlow offre les trois principales couches RNN : RNN simple, GRU et LSTM.</li>
    <li> <strong>Couche dense:</strong> Cette couche finale prend la sortie du RNN et applique une classe à la phrase, c'est-à-dire un 1 si la critique est positive.
    </li>
</ul>

In [ ]:
from keras.models import Sequential
from keras.layers import Bidirectional, SimpleRNN, Dense, Embedding

model = Sequential()

model.add(Embedding(max_features, 8, input_length=maxlen))
model.add(Bidirectional(SimpleRNN(64)))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='rmsprop', 
              loss = 'binary_crossentropy', 
              metrics=['accuracy']) 

model.summary()

#### 2.1.3 Entraîner le modèle

In [ ]:
epochs = 10
batch_size = 32

history = model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2)

#### 2.1.3 Suavegarder le modèle

In [11]:
import os 
import numpy as np 

# 💾 Création du dossier général pour enregistrer les résultats
SAVE_DIR = "RNN_resultats"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, "Embedding_SimpleRNN_model.h5"))
np.save(os.path.join(SAVE_DIR, "Embedding_SimpleRNN_model_history.npy"), history.history)

#### 2.1.4 Plot metrics

In [ ]:
from utils import plot_all_metrics

test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

plot_all_metrics({"Embedding_SimpleRNN": history}, SAVE_DIR, "model_loss_plot_Embedding_SimpleRNN.png")

### 2.2 Utiliser un modèle pré-entraîné 'GloVe'

In [ ]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.initializers import Constant


max_features = 10000  # mots les plus fréquents
maxlen = 20  # taille max des séquences

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

#### 2.2.1 Charger les vecteurs GloVe

à télécharger depuis : https://nlp.stanford.edu/projects/glove/

In [ ]:
from zipfile import ZipFile

zf = ZipFile('glove.6B.zip', 'r')
zf.extractall('glove.6B')
zf.close()

In [ ]:
import os 

embedding_dim = 100
glove_path = os.path.join('glove.6B', 'glove.6B.100d.txt') 

embedding_index = {}
with open(glove_path,'r',encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coeffs = np.asarray(values[1:], dtype='float32')
        # Transformer chaque mot en un vecteur de dimensions 100.
        embedding_index[word] = coeffs

#### 2.2.3 Créer la matrice embedding

In [4]:
word_index = imdb.get_word_index()

embedding_matrix = np.zeros((max_features, embedding_dim))
for word, i in word_index.items():
    if i < max_features:
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

#### 2.2.4 Construire le modèle

In [ ]:
model = Sequential()
model.add(Embedding(
    input_dim=max_features,
    output_dim=embedding_dim,
    embeddings_initializer=Constant(embedding_matrix),
    input_length=maxlen,
    trainable=False
))
model.add(SimpleRNN(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

#### 2.2.5 Entraîner le modèle

In [ ]:
epochs=5
batch_size=128

history = model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2)

#### 2.2.6 Sauvegarder le modèle 

In [15]:
SAVE_DIR = "RNN_resultats"
model.save(os.path.join(SAVE_DIR, "GloVe_SimpleRNN_model.h5"))
np.save(os.path.join(SAVE_DIR, "GloVe_SimpleRNN_model_history.npy"), history.history)

c:\Users\gbencheikh\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [16]:
with open(os.path.join(SAVE_DIR, "GloVe_SimpleRNN_model_architecture.json"), "w") as f:
    f.write(model.to_json())

# Sauvegarde des poids
model.save_weights(os.path.join(SAVE_DIR, "GloVe_SimpleRNN_model_weights.h5"))

#### 2.2.7 Plot metrics

In [ ]:
from utils import plot_all_metrics

plot_all_metrics({"GloVe_SimpleRNN_model": history}, SAVE_DIR, "GloVe_SimpleRNN_model.png")

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'\nTest Accuracy: {test_acc:.3f}')

### 2.3 Tester les modèles

#### 2.1.6 Tester sur une phrase personnalisée (externe au dataset)

Là, il faut faire un petit prétraitement :

* convertir les mots en indices via le dictionnaire de mots word_index

* paddiner pour qu'elle soit de longueur 20

In [ ]:
from utils import encode_review
from keras.models import load_model
import os 

# Exemple : phrase à tester
text = "the movie was so boring"
encoded_text = encode_review(text)

In [ ]:
model_embedding = load_model(os.path.join("RNN_resultats", "Embedding_SimpleRNN_model.h5"))
prediction_embedding = model_embedding.predict(encoded_text, verbose=False)
print("Prediction embedding:", "positif" if (prediction_embedding[0][0] > 0.5) else "negatif")

In [19]:
from tensorflow.keras.models import model_from_json

# 1. Charger l'architecture du modèle
with open(os.path.join(SAVE_DIR, "GloVe_SimpleRNN_model_architecture.json"), "r") as f:
    model_json = f.read()

model_GloVe = model_from_json(model_json)

# 2. Charger les poids
model_GloVe.load_weights(os.path.join(SAVE_DIR, "GloVe_SimpleRNN_model_weights.h5"))

# 3. Compiler à nouveau (obligatoire pour prédire ou entraîner)
model_GloVe.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 4. Charger l’historique si nécessaire
history = np.load(os.path.join(SAVE_DIR, "GloVe_SimpleRNN_model_history.npy"), allow_pickle=True).item()

# Prêt à prédire ou réentraîner

prediction_GloVe = model_GloVe.predict(encoded_text, verbose=False)
print("Prediction GloVe:", "positif" if (prediction_GloVe[0][0] > 0.5) else "negatif")

Cause: Unable to locate the source code of <function Model.make_predict_function.<locals>.predict_function at 0x0000027CD7FE5120>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: Unable to locate the source code of <function Model.make_predict_function.<locals>.predict_function at 0x0000027CD7FE5120>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Orig